# Notebook 10: Dataset Expansion & Activation Re-extraction

**Purpose:** Expand the IRIS dataset from 1,000 to ~5,000 balanced examples,
then re-extract GPT-2 activations and SAE features for the expanded dataset.

**Prerequisites:** Pre-trained SAE checkpoint (`sae_d6144_lambda1e-04.pt`) on Google Drive.

**Runtime:** Colab GPU (T4), ~15 minutes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -r /content/drive/MyDrive/iris/requirements.txt -q

In [ ]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/iris')

from src.utils.helpers import set_seed, get_device
set_seed(42)
device = get_device()

## Step 1: Expand the Dataset

Pull from HuggingFace datasets to grow from 1,000 to ~5,000 balanced examples.
This adds more injection categories and diverse normal prompts.

In [ ]:
!pip install datasets -q

In [ ]:
# Run the expansion script
%run /content/drive/MyDrive/iris/scripts/expand_dataset.py

In [ ]:
# Verify the expanded dataset
from src.data.dataset import IrisDataset
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/iris')
expanded_path = DRIVE_ROOT / 'data' / 'processed' / 'iris_dataset_expanded.json'

dataset = IrisDataset.load(expanded_path)
dataset.summary()

## Step 2: Extract Activations for Expanded Dataset

Run GPT-2 on all prompts and extract residual stream activations at all 12 layers.
This gives us the raw representations needed for detection and SAE decomposition.

In [ ]:
import torch
import numpy as np
from src.model.transformer import load_model, extract_activations

# Load GPT-2
model = load_model(device)

In [ ]:
# Tokenize the expanded dataset
formatted_prompts = dataset.format_prompts()
print(f'Tokenizing {len(formatted_prompts)} prompts...')

tokens = model.tokenizer(
    formatted_prompts,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors='pt',
)
input_ids = tokens['input_ids']
attention_mask = tokens['attention_mask']
print(f'Token shape: {input_ids.shape}')

In [ ]:
# Extract activations at all 12 layers
# This takes ~5 minutes on a T4 for 5000 prompts
activations = extract_activations(
    model, input_ids, attention_mask,
    layers=list(range(12)),
    batch_size=32,
)

# Save to Drive
save_dict = {f'layer_{i}': activations[i] for i in range(12)}
save_dict['labels'] = np.array(dataset.labels)

save_path = DRIVE_ROOT / 'checkpoints' / 'expanded_activations.npz'
np.savez_compressed(str(save_path), **save_dict)
print(f'Saved activations to {save_path}')
print(f'File size: {save_path.stat().st_size / 1e6:.1f} MB')

## Step 3: Compute SAE Features for Expanded Dataset

Run the trained SAE encoder on layer-0 activations to get the 6144-dim
sparse feature vectors for every prompt.

In [ ]:
from src.sae.architecture import SparseAutoencoder
from src.utils.helpers import load_checkpoint
from src.analysis.features import compute_feature_activations, compute_sensitivity_scores

# Load the trained SAE (8x expansion, lambda=1e-4)
sae = SparseAutoencoder(d_input=768, expansion_factor=8, sparsity_coeff=1e-4)
ckpt_path = DRIVE_ROOT / 'checkpoints' / 'sae_d6144_lambda1e-04.pt'
load_checkpoint(ckpt_path, sae, device=device)
sae = sae.to(device)
sae.eval()

print(f'SAE loaded: {sae.d_input} -> {sae.d_sae}')

In [ ]:
# Compute SAE features on layer-0 activations
layer0_acts = activations[0]  # shape: (N, 768)
feature_matrix = compute_feature_activations(sae, layer0_acts, device=device)
print(f'Feature matrix shape: {feature_matrix.shape}')

# Compute sensitivity scores
labels = np.array(dataset.labels)
sensitivity = compute_sensitivity_scores(feature_matrix, labels)

# Save
np.save(str(DRIVE_ROOT / 'checkpoints' / 'expanded_feature_matrix.npy'), feature_matrix)
np.save(str(DRIVE_ROOT / 'checkpoints' / 'expanded_sensitivity_scores.npy'), sensitivity)
print('Saved feature matrix and sensitivity scores')

## Step 4: Quick Validation

Run a quick detection comparison to verify the expanded dataset works.

In [ ]:
from sklearn.model_selection import train_test_split
from src.analysis.detection import run_detection_comparison

# 70/30 split
n = len(labels)
indices = np.arange(n)
train_idx, test_idx = train_test_split(
    indices, train_size=0.7, stratify=labels, random_state=42
)

texts = dataset.texts
results, preds = run_detection_comparison(
    train_texts=[texts[i] for i in train_idx],
    train_labels=labels[train_idx],
    test_texts=[texts[i] for i in test_idx],
    test_labels=labels[test_idx],
    train_activations=layer0_acts[train_idx],
    test_activations=layer0_acts[test_idx],
    train_features=feature_matrix[train_idx],
    test_features=feature_matrix[test_idx],
)

In [ ]:
# Save quick-check results
import json

results_path = DRIVE_ROOT / 'results' / 'metrics' / 'c3_expanded_quick_check.json'
results_path.parent.mkdir(parents=True, exist_ok=True)
results_path.write_text(json.dumps(results, indent=2))
print(f'Saved to {results_path}')

print('\n--- Dataset expansion complete! ---')
print(f'Dataset: {len(dataset)} examples')
print(f'Activations: {layer0_acts.shape}')
print(f'SAE features: {feature_matrix.shape}')
print(f'Sensitivity scores: {sensitivity.shape}')